# Proyecto 03: Modificando el Marco de Referencia

## 0. Código y Funciones Auxiliares

In [213]:
# Import de librerías para la realización del proyecto

from pathlib import Path
from functools import reduce
from typing import Callable

import cv2
import numpy as np
import matplotlib.pyplot as plt

### 0.1 Carga de Imágenes

In [210]:
def load_images_dataset():
    """
    Función para cargar/leer las imágenes del dataset del proyecto. 
    Se cargan usando OpenCV y convertidas al formato RGB.

    Devuelve dos listas, una con las imágenes cargadas y otra 
    con los nombres de las imágenes.
    """

    base_path = Path("./dataset")
    images = []
    filenames =  []
    for path in base_path.iterdir():
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        images.append(image)
        filenames.append(path.name[-5])
    return images, filenames

In [211]:
Path("./results").mkdir(exist_ok=True)
images, filenames = load_images_dataset()

### 0.2 Transformaciones Afines

In [432]:
def translation_matrix(
        x_move: float = 0,
        y_move: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de traslación. Toma 
    de entrada el movimiento 
    horizontal y vertical.
    """

    matrix = np.array([
        [1, 0, x_move],
        [0, 1, y_move],
        [0, 0, 1]
    ], dtype=float)
    return matrix

def inverse_translation_matrix(
        x_move: float = 0,
        y_move: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de traslación inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    return translation_matrix(-x_move, -y_move)

In [434]:
def rotation_matrix(angle: float) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación. Toma 
    de entrada el ángulo de 
    rotación.
    """

    cos_angle = np.cos(angle)
    sin_angle = np.sin(angle)
    matrix = np.array([
        [cos_angle, sin_angle, 0],
        [-sin_angle, cos_angle, 0],
        [0, 0, 1]
    ], dtype=float)
    return matrix

def inverse_rotation_matrix(angle: float) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    return rotation_matrix(-angle)

In [438]:
def centered_rotation_matrix(
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación centrada 
    respecto al centro de la 
    imagen. Toma de entrada el 
    centro de la imagen y el 
    ángulo de rotation.
    """

    rotation = rotation_matrix(angle)
    transformed_center = rotation@center
    adjust = (center-transformed_center)
    centered_rotation = translate_matrix(adjust[0], adjust[1])@rotation
    return centered_rotation

def inverse_centered_rotation_matrix(
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación centrada inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    return centered_rotation_matrix(center, -angle)

In [441]:
def reflection_matrix(
        reflect_x: bool = False,
        reflect_y: bool = False,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de reflexión sobre los ejes. Toma 
    de entrada si se aplica la reflexión 
    individualmente sobre los ejes x y y.
    """

    value_x = -1 if reflect_x else 1
    value_y = -1 if reflect_y else 1
    reflection = np.array([
        [value_y, 0, 0],
        [0, value_x, 0],
        [0, 0, 1],
    ])
    return reflection

def inverse_reflection_matrix(
        reflect_x: bool = False,
        reflect_y: bool = False,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de reflexión inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    return reflection_matrix(reflect_x, reflect_y)

In [444]:
def scaling_matrix(
        scale_x: float = 1,
        scale_y: float = 1,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de escala sobre los ejes. 
    Toma de entrada las nuevas escalas 
    de los ejes x y y.
    """

    scaling = np.array([
        [scale_x, 0, 0],
        [0, scale_y, 0],
        [0, 0, 1],
    ])
    return scaling

def inverse_scaling_matrix(
        scale_x: float = 1,
        scale_y: float = 1,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de escalmiento inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    return scaling_matrix(1/scale_x, 1/scale_y)

In [449]:
def shearing_matrix(
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de cizallamiento sobre los ejes. 
    Toma de entrada los desplazamientos 
    de los ejes x y y.
    """

    shearing = np.array([
        [1, shift_x, 0],
        [shift_y, 1, 0],
        [0, 0, 1]
    ])
    return shearing

def inverse_shearing_matrix(
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de cizallamiento inversa 
    usando los mismos parámetros 
    que en la matriz original.
    """

    determinant_shearing = 1-shift_x*shift_y
    inverse_shearing = np.array([
        [1, -shift_x, 0],
        [-shift_y, 1, 0],
        [0, 0, determinant_shearing]
    ])
    return inverse_shearing/determinant_shearing

In [450]:
def affine_transformation_matrix(
        x_move: float = 0,
        y_move: float = 0,
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
        reflect_x: bool = False,
        reflect_y: bool = False,
        scale_x: float = 1, 
        scale_y: float = 1,
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la matriz general 
    de una transformación afín. Toma de 
    parámetros los definidos para cada 
    matriz de transformación individualmente.
    """

    transformations = [
        translation_matrix(x_move, y_move),
        centered_rotation_matrix(center, angle),
        reflection_matrix(reflect_x, reflect_y),
        scaling_matrix(scale_x, scale_y),
        shearing_matrix(shift_x, shift_y),
    ]
    return reduce(lambda t_1, t_2 : t_1@t_2, transformations)

def inverse_affine_transformation_matrix(
        x_move: float = 0,
        y_move: float = 0,
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
        reflect_x: bool = False,
        reflect_y: bool = False,
        scale_x: float = 1, 
        scale_y: float = 1,
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la matriz general 
    de una transformación afín inversa. Toma de 
    parámetros los mismos que los definidos 
    para cada matriz inversa y equivalentes 
    a la original.
    """

    inverse_transformations = [
        inverse_shearing_matrix(shift_x, shift_y),
        inverse_scaling_matrix(scale_x, scale_y),
        inverse_reflection_matrix(reflect_x, reflect_y),
        inverse_centered_rotation_matrix(center, angle),
        inverse_translation_matrix(x_move, y_move),
    ]
    return reduce(lambda t_1, t_2 : t_1@t_2, inverse_transformations)

### 0.3 Interpolación Bilineal

### 0.4 Visualizar Transformaciones

In [214]:
def save_plot(
        create_plot: Callable[[np.ndarray, str], tuple[plt.Figure, str | None]]
    ) -> Callable:
    """
    Función para guardar un plot diseñado o 
    creado. Se espera que la función decorada 
    devuelva el objeto Figure y el nombre de 
    la figura.

    Devuelve el plot creado.
    """

    def saver(image: np.ndarray, *args, **kwargs) -> plt.Figure:
        fig, fig_name = create_plot(image, *args, **kwargs)
        if fig_name: fig.savefig(f"./results/{fig_name}.jpg", dpi=300, bbox_inches="tight")
        return fig
    
    return saver

## 1. Imagen 1